# Notebook 3: Protein Feature Extraction (ProtT5)

**Purpose:** Extract residue-level protein sequence features using the pre-trained
ProtT5 language model (`Rostlab/prot_t5_xl_uniref50`).

**Input:** `dataset/bisai/target_info.csv` — protein UniProt IDs and amino acid sequences.

**Output:** One `.npy` file per protein saved to Google Drive at
`EviDTI_Protein_Features_Unpooled/`. Each file contains a matrix of shape `(L, 1024)`,
where `L` is the sequence length.

**Method:** ProtT5 encodes each amino acid into a 1024-dimensional embedding.
Features are saved at residue level (unpooled) and subsequently processed by the
Light Attention module during model training.

> **Note:** ProtT5 is an 11 GB model. First-time download takes ~10 minutes.
> Half-precision (fp16) inference is used to fit within Colab T4 GPU memory.
> If the session crashes mid-run, simply re-run Cell 4 — already-processed proteins
> are automatically skipped.

## Step 0: Install Dependencies, Mount Drive and Configure Paths

In [ ]:
# Install required dependencies
!pip install -q transformers sentencepiece accelerate torch

from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

# ================= Configure paths =================
# Path to protein sequence CSV file
csv_path = "/content/drive/MyDrive/EviDTI-main/dataset/bisai/target_info.csv"
# Output directory: one .npy file per protein
save_dir = "/content/drive/MyDrive/EviDTI_Protein_Features_Unpooled"

# Create output directory if it does not exist
os.makedirs(save_dir, exist_ok=True)
print(f"Output directory ready: {save_dir}")

## Step 1: Extract Project Files

In [ ]:
import shutil
import os

# Update zip_path to match the actual backup filename in your Google Drive
zip_path = "/content/drive/MyDrive/EviDTI_Feature_Extraction_Backup_20260303_1712.zip"
extract_path = "/content/EviDTI-main"

print("Extracting project files...")
if not os.path.exists(f"{extract_path}/dataset/bisai/target_info.csv"):
    shutil.unpack_archive(zip_path, extract_path)
    print("Extraction complete.")
else:
    print("Directory already exists, skipping extraction.")

## Step 2: Load and Clean Protein Sequences

In [ ]:
import pandas as pd
import re

print("Loading protein sequences...")

# Read target_info.csv from the extracted local directory
csv_path = "/content/EviDTI-main/dataset/bisai/target_info.csv"
# Save extracted features to Google Drive
save_dir = "/content/drive/MyDrive/EviDTI_Protein_Features_Unpooled"
os.makedirs(save_dir, exist_ok=True)

df = pd.read_csv(csv_path, names=['id', 'input'])

# Clean sequences: strip whitespace and replace rare amino acids with X
df['input_fixed'] = ["".join(str(seq).split()) for seq in df['input']]
df['input_fixed'] = [re.sub(r"[UZOB]", "X", seq) for seq in df['input_fixed']]

# Convert to ProtT5 input format (list of characters per sequence)
dataset_ids = df['id'].tolist()
dataset_seqs = [list(seq) for seq in df['input_fixed']]

print(f"Loaded and cleaned {len(dataset_ids)} protein sequences.")
print(f"Sample ID: {dataset_ids[0]}, sequence start: {dataset_seqs[0][:10]}")

## Step 3: Load ProtT5 Model

In [ ]:
import torch
from transformers import T5EncoderModel, T5Tokenizer

print("Loading ProtT5 model from HuggingFace (first run: ~10 min download)...")
model_name = "Rostlab/prot_t5_xl_uniref50"

# Download tokenizer and model
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name)

# Detect and assign GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = model.to(device)
model = model.eval()  # Switch to inference mode

# Use half precision (fp16) to reduce GPU memory usage
if torch.cuda.is_available():
    model = model.half()
    print("Half-precision (fp16) enabled.")

## Step 4: Extract Residue-Level Features

Iterates over all proteins and saves each as an individual `.npy` file.
Supports checkpoint resumption: already-processed files are skipped automatically.

In [ ]:
import numpy as np
import gc
import os
from tqdm.auto import tqdm

print(f"Extracting residue-level features. Saving to: {save_dir}")

for i in tqdm(range(len(dataset_ids))):
    prot_id = str(dataset_ids[i]).strip()
    seq_list = dataset_seqs[i]
    save_path = os.path.join(save_dir, f"{prot_id}.npy")

    # Skip proteins already processed (supports resumption after session crash)
    if os.path.exists(save_path):
        continue

    with torch.no_grad():
        # 1. Tokenise: join amino acids with spaces as required by ProtT5
        seq_string = " ".join(seq_list)
        ids = tokenizer(seq_string, add_special_tokens=True, padding=True, return_tensors="pt")

        # 2. Run inference
        embedding = model(input_ids=ids['input_ids'].to(device))[0]

        # 3. Extract embeddings and remove trailing EOS token -> shape: (L, 1024)
        emb_numpy = embedding[0].detach().cpu().numpy()[0:-1]

        # 4. Save as individual file
        np.save(save_path, emb_numpy)

    # 5. Free memory
    del ids, embedding, emb_numpy

    # Full GPU cache clear every 50 proteins
    if i % 50 == 0:
        torch.cuda.empty_cache()
        gc.collect()

print("All protein features extracted and saved to Google Drive.")